# CWP Top-K Reactivity Enrichment (v1)

This notebook demonstrates practical downselection value by ranking CWP peptides, then plotting cumulative proportion reactive vs selected peptides (Top-K), with a full-library baseline.

v1 uses current pulled artifacts only (`PredOnes` from `peptide_comparison.csv`) and does not modify pipeline behavior.

In [ ]:
from math import erf, sqrt
from pathlib import Path
import hashlib
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pandas.testing import assert_frame_equal

In [ ]:
# Parameterized config
base_dir = Path("localdata/evals/cocci_eval/seeded_runs")
model_id = "flagship2"
set_id = "set_01"
mode = "combined"

k_start = 500
k_step = 10
tie_policy = "strict_deterministic"

alpha = 0.05
z_95 = 1.959963984540054

In [ ]:
def _resolve_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    if (cwd / "pyproject.toml").exists():
        return cwd
    if (cwd.parent / "pyproject.toml").exists():
        return cwd.parent
    raise RuntimeError("Could not locate repository root containing pyproject.toml")


repo_root = _resolve_repo_root()
run_dir = repo_root / base_dir / model_id / set_id / mode
compare_csv = run_dir / "peptide_compare" / "peptide_comparison.csv"
compare_summary_json = run_dir / "peptide_compare" / "peptide_comparison_summary.json"
output_dir = repo_root / "notebooks" / "artifacts"
output_dir.mkdir(parents=True, exist_ok=True)

if tie_policy != "strict_deterministic":
    raise ValueError(f"Unsupported tie_policy for v1: {tie_policy}")

if not compare_csv.exists():
    raise FileNotFoundError(f"Missing peptide comparison CSV: {compare_csv}")
if not compare_summary_json.exists():
    raise FileNotFoundError(f"Missing peptide comparison summary JSON: {compare_summary_json}")

raw_df = pd.read_csv(compare_csv)
summary_payload = json.loads(compare_summary_json.read_text(encoding="utf-8"))

required_cols = {"CodeName", "TrueClass", "PredOnes", "PeptideLen"}
missing_cols = sorted(required_cols - set(raw_df.columns))
assert not missing_cols, f"Missing required columns: {missing_cols}"

classes = set(raw_df["TrueClass"].astype(str).str.lower().unique().tolist())
assert "reactive" in classes and "nonreactive" in classes, (
    f"TrueClass must contain reactive and nonreactive. Got: {classes}"
)

print(f"repo_root: {repo_root}")
print(f"run_dir: {run_dir}")
print(f"rows: {len(raw_df):,}")
print(f"class_counts: {raw_df['TrueClass'].value_counts().to_dict()}")

In [ ]:
def _normal_cdf(x: float) -> float:
    return 0.5 * (1.0 + erf(x / sqrt(2.0)))


def wilson_ci(successes: int, n: int, z: float = z_95) -> tuple[float, float]:
    if n <= 0:
        return float("nan"), float("nan")
    p = successes / n
    denom = 1.0 + (z**2 / n)
    center = (p + (z**2 / (2.0 * n))) / denom
    margin = z * sqrt((p * (1.0 - p) + (z**2 / (4.0 * n))) / n) / denom
    lo = max(0.0, center - margin)
    hi = min(1.0, center + margin)
    return lo, hi


def one_sided_two_proportion_ztest(
    x1: int,
    n1: int,
    x2: int,
    n2: int,
    h1: str = "greater"
) -> tuple[float, float]:
    if h1 != "greater":
        raise ValueError("v1 supports only h1='greater'")
    if n1 <= 0 or n2 <= 0:
        return float("nan"), float("nan")

    p1 = x1 / n1
    p2 = x2 / n2
    p_pool = (x1 + x2) / (n1 + n2)
    var = p_pool * (1.0 - p_pool) * ((1.0 / n1) + (1.0 / n2))
    if var <= 0.0:
        return float("nan"), float("nan")

    z = (p1 - p2) / sqrt(var)
    p_one_sided = 1.0 - _normal_cdf(z)
    return z, p_one_sided


def build_ranked_df(df: pd.DataFrame) -> pd.DataFrame:
    ranked = df.copy()
    ranked["CodeName"] = ranked["CodeName"].astype(str)
    ranked["TrueClass"] = ranked["TrueClass"].astype(str).str.lower()
    ranked["score_sum"] = pd.to_numeric(ranked["PredOnes"], errors="raise")
    ranked["is_reactive"] = (ranked["TrueClass"] == "reactive").astype(int)
    ranked = ranked.sort_values(
        ["score_sum", "CodeName"],
        ascending=[False, True],
        kind="mergesort"
    ).reset_index(drop=True)
    return ranked


def build_k_grid(n_total: int, start: int, step: int) -> list[int]:
    if n_total < start:
        raise ValueError(f"n_total={n_total} is smaller than k_start={start}")
    if step <= 0:
        raise ValueError("k_step must be positive")

    ks = list(range(start, n_total + 1, step))
    if ks[-1] != n_total:
        ks.append(n_total)
    return ks


def compute_topk_metrics(
    ranked_df: pd.DataFrame,
    k_values: list[int],
    baseline_prop: float,
    reactive_total: int,
    n_total: int
) -> pd.DataFrame:
    cum_reactive = ranked_df["is_reactive"].cumsum().to_numpy(dtype=np.int64)
    rows: list[dict[str, float | int | str | None]] = []

    for k in k_values:
        reactive_topk = int(cum_reactive[k - 1])
        prop_topk = float(reactive_topk / k)
        ci_lo, ci_hi = wilson_ci(reactive_topk, k)
        er = float(prop_topk / baseline_prop) if baseline_prop > 0.0 else float("nan")

        # Requested v1 handling: mark known edge case K=N as non-computable.
        z_stat = float("nan")
        pval = float("nan")
        pval_note = "ok"
        if k == n_total:
            pval_note = "non-computable: k equals full library size"
        else:
            z_stat, pval = one_sided_two_proportion_ztest(
                x1=reactive_topk,
                n1=k,
                x2=reactive_total,
                n2=n_total,
                h1="greater"
            )
            if not np.isfinite(pval):
                pval_note = "non-computable: invalid variance or counts"

        rows.append(
            {
                "k_selected": int(k),
                "reactive_topk": int(reactive_topk),
                "prop_topk": float(prop_topk),
                "prop_topk_ci_lo": float(ci_lo),
                "prop_topk_ci_hi": float(ci_hi),
                "reactive_total": int(reactive_total),
                "n_total": int(n_total),
                "prop_full": float(baseline_prop),
                "enrichment_ratio": float(er),
                "z_stat_one_sided": float(z_stat),
                "pvalue_one_sided": float(pval),
                "pvalue_note": str(pval_note),
            }
        )

    return pd.DataFrame(rows)


def metrics_hash(df: pd.DataFrame) -> str:
    csv_payload = df.to_csv(index=False, float_format="%.12g")
    return hashlib.sha256(csv_payload.encode("utf-8")).hexdigest()

In [ ]:
ranked_df = build_ranked_df(raw_df)
n_total = int(len(ranked_df))
reactive_total = int(ranked_df["is_reactive"].sum())
prop_full = float(reactive_total / n_total)

summary_n_rows = int(summary_payload.get("n_rows_compared"))
summary_n_reactive = int(summary_payload.get("n_reactive"))
summary_prop = float(summary_n_reactive / summary_n_rows)

assert summary_n_rows == n_total, (
    f"Row mismatch between CSV ({n_total}) and summary JSON ({summary_n_rows})"
)
assert summary_n_reactive == reactive_total, (
    f"Reactive count mismatch between CSV ({reactive_total}) and summary JSON ({summary_n_reactive})"
)
assert np.isclose(prop_full, summary_prop, rtol=0.0, atol=1e-12), (
    f"Baseline proportion mismatch: csv={prop_full} json={summary_prop}"
)

k_values = build_k_grid(n_total=n_total, start=k_start, step=k_step)
assert k_values[0] == k_start, "K grid must start at k_start"
assert k_values[-1] == n_total, "K grid must end at N"

if len(k_values) > 1:
    diffs = np.diff(np.asarray(k_values, dtype=np.int64))
    if n_total % k_step == k_start % k_step:
        assert np.all(diffs == k_step), "K grid increments must be exactly k_step"
    else:
        assert np.all(diffs[:-1] == k_step), (
            "All interior K increments must be k_step when final K is appended as N"
        )

metrics_df = compute_topk_metrics(
    ranked_df=ranked_df,
    k_values=k_values,
    baseline_prop=prop_full,
    reactive_total=reactive_total,
    n_total=n_total,
)

assert np.isclose(metrics_df.iloc[-1]["prop_topk"], prop_full, atol=1e-12), (
    "Final Top-K point (K=N) must equal full-library proportion"
)
assert ((metrics_df["prop_topk_ci_lo"] >= 0.0) & (metrics_df["prop_topk_ci_hi"] <= 1.0)).all(), (
    "Wilson CI bounds must stay within [0,1]"
)
assert (metrics_df["prop_topk_ci_lo"] <= metrics_df["prop_topk_ci_hi"]).all(), (
    "Wilson CI lower bound must not exceed upper bound"
)

# Reproducibility check: deterministic rerun must match.
ranked_df_rerun = build_ranked_df(raw_df)
metrics_df_rerun = compute_topk_metrics(
    ranked_df=ranked_df_rerun,
    k_values=k_values,
    baseline_prop=prop_full,
    reactive_total=reactive_total,
    n_total=n_total,
)

assert_frame_equal(ranked_df.head(20), ranked_df_rerun.head(20), check_dtype=False)
assert_frame_equal(ranked_df.tail(20), ranked_df_rerun.tail(20), check_dtype=False)

hash_1 = metrics_hash(metrics_df)
hash_2 = metrics_hash(metrics_df_rerun)
assert hash_1 == hash_2, "Deterministic rerun failed: metrics hash mismatch"

print(f"N total peptides: {n_total:,}")
print(f"Reactive peptides: {reactive_total:,}")
print(f"Full-library reactive proportion: {prop_full:.8f}")
print(f"Metrics rows: {len(metrics_df):,}")
print(f"Deterministic hash: {hash_1}")

In [ ]:
checkpoint_targets = [500, 1000, 2500, 5000, 10000]
checkpoint_targets.extend(range(20000, n_total + 1, 10000))
checkpoint_targets.append(n_total)

checkpoint_targets = sorted(set(int(k) for k in checkpoint_targets if k <= n_total))
checkpoints_df = metrics_df[metrics_df["k_selected"].isin(checkpoint_targets)].copy()

display_cols = [
    "k_selected",
    "reactive_topk",
    "prop_topk",
    "prop_topk_ci_lo",
    "prop_topk_ci_hi",
    "prop_full",
    "enrichment_ratio",
    "z_stat_one_sided",
    "pvalue_one_sided",
    "pvalue_note",
]

checkpoints_df[display_cols].head(20)

In [ ]:
set_token = set_id.replace("_", "")
artifact_stem = f"cwp_topk_enrichment_{model_id}_{set_token}"
plot_png = output_dir / f"{artifact_stem}.png"
plot_svg = output_dir / f"{artifact_stem}.svg"
csv_all = output_dir / f"{artifact_stem}.csv"
csv_ckpt = output_dir / f"{artifact_stem}_checkpoints.csv"

for out_path in (plot_png, plot_svg, csv_all, csv_ckpt):
    if not out_path.resolve().is_relative_to(output_dir.resolve()):
        raise RuntimeError(f"Refusing to write outside notebooks/artifacts: {out_path}")

x = metrics_df["k_selected"].to_numpy(dtype=float)
y = metrics_df["prop_topk"].to_numpy(dtype=float)
y_lo = metrics_df["prop_topk_ci_lo"].to_numpy(dtype=float)
y_hi = metrics_df["prop_topk_ci_hi"].to_numpy(dtype=float)
er = metrics_df["enrichment_ratio"].to_numpy(dtype=float)

fig, (ax1, ax2) = plt.subplots(
    2,
    1,
    figsize=(11, 8),
    sharex=True,
    gridspec_kw={"height_ratios": [2.2, 1.2]}
)

ax1.plot(x, y, color="#1f77b4", linewidth=2.0, label="Top-K reactive proportion")
ax1.fill_between(x, y_lo, y_hi, color="#1f77b4", alpha=0.20, label="95% CI")
ax1.axhline(prop_full, color="#d62728", linestyle="--", linewidth=1.8, label="Full-library baseline")
ax1.set_ylabel("Proportion Reactive")
ax1.set_title(f"CWP Top-K Enrichment ({model_id} {set_id})", fontsize=20)
ax1.grid(axis="y", linestyle="--", alpha=0.4)
ax1.legend(loc="best")

ax2.plot(x, er, color="#2ca02c", linewidth=1.8, label="Enrichment ratio (Top-K / full)")
ax2.axhline(1.0, color="#7f7f7f", linestyle=":", linewidth=1.6)
ax2.set_ylabel("Enrichment Ratio")
ax2.set_xlabel("Number of Selected Peptides (K)")
ax2.grid(axis="y", linestyle="--", alpha=0.4)
ax2.legend(loc="best")

fig.tight_layout()
fig.savefig(plot_png, dpi=300, bbox_inches="tight")
fig.savefig(plot_svg, dpi=300, bbox_inches="tight")
plt.show()

metrics_df.to_csv(csv_all, index=False)
checkpoints_df.to_csv(csv_ckpt, index=False)

print(f"Saved all-metrics CSV: {csv_all}")
print(f"Saved checkpoints CSV: {csv_ckpt}")
print(f"Saved figure PNG: {plot_png}")
print(f"Saved figure SVG: {plot_svg}")

In [ ]:
# Compact statistical summary at practical K checkpoints.
summary_table = checkpoints_df[[
    "k_selected",
    "reactive_topk",
    "prop_topk",
    "prop_topk_ci_lo",
    "prop_topk_ci_hi",
    "prop_full",
    "enrichment_ratio",
    "pvalue_one_sided",
    "pvalue_note",
]].copy()

summary_table.sort_values("k_selected").reset_index(drop=True).tail(25)